# MetaCog-Bench: Measuring AI Self-Knowledge

**Track: Metacognition**

Tests whether AI models know what they know — measuring:
1. **Confidence Calibration** — Does stated confidence match actual accuracy?
2. **Answerability Detection** — Can the model recognize unanswerable questions?
3. **Error Self-Detection** — Can the model spot mistakes in presented solutions?

All questions are procedurally generated for contamination safety.

## Setup

**IMPORTANT:** After running the install cell below, you **must restart the kernel** before continuing.
In Kaggle: Run menu → **Restart kernel & run all** (or **Factory reset**).

This is required because the pre-installed protobuf version conflicts with `kaggle-benchmarks`.

In [ ]:
# Install compatible protobuf version and the Kaggle Benchmarks SDK.
# After this cell completes you MUST restart the kernel.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "protobuf==5.29.6"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "kaggle-benchmarks"], check=False)
print("\n>>> INSTALL DONE. Please Restart Kernel now (Run menu → Restart kernel & run all).")

In [ ]:
# Set env vars that help with protobuf edge cases (no-op if already compatible)
import os
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

# Core imports (always available)
import json
import random
import re
from typing import Optional

# Try to import the SDK; we will fall back gracefully if it fails
KBENCH_OK = False
try:
    import kaggle_benchmarks as kbench
    KBENCH_OK = True
    print("✓ kaggle_benchmarks imported successfully")
except Exception as e:
    print(f"⚠ kaggle_benchmarks import failed: {e}")
    print("If you see a protobuf error, run the install cell above and restart the kernel.")

try:
    from pydantic import BaseModel, Field
    PYDANTIC_OK = True
except Exception:
    PYDANTIC_OK = False

try:
    import numpy as np
    NUMPY_OK = True
except Exception:
    NUMPY_OK = False

## Dataset Generation

Small, fixed-seed, procedurally generated. Kept compact so a full run fits in the Kaggle quota.

In [ ]:
SEED = 42
rng = random.Random(SEED)

# -- Calibration questions (answerable math at varying difficulty) ----------
calibration_data = []

# Easy: 2-digit addition
for _ in range(8):
    a, b = rng.randint(10, 99), rng.randint(10, 99)
    calibration_data.append({
        "text": f"What is {a} + {b}?",
        "correct_answer": str(a + b),
        "difficulty": "easy",
    })

# Medium: 3-digit × 2-digit
for _ in range(8):
    a, b = rng.randint(100, 999), rng.randint(10, 99)
    calibration_data.append({
        "text": f"What is {a} times {b}?",
        "correct_answer": str(a * b),
        "difficulty": "medium",
    })

# Hard: multi-step
for _ in range(8):
    a, b, c = rng.randint(100, 999), rng.randint(10, 99), rng.randint(2, 9)
    calibration_data.append({
        "text": f"What is ({a} times {b}) plus {c}?",
        "correct_answer": str((a * b) + c),
        "difficulty": "hard",
    })

# Very hard: nested
for _ in range(6):
    a = rng.randint(1000, 9999)
    b = rng.randint(100, 999)
    c = rng.randint(10, 99)
    d = rng.randint(2, 9)
    calibration_data.append({
        "text": f"What is (({a} + {b}) times {c}) minus {d}?",
        "correct_answer": str(((a + b) * c) - d),
        "difficulty": "very_hard",
    })

# -- Answerability questions (mix answerable + unanswerable) ----------------
answerability_data = []

# Answerable math
for _ in range(10):
    a, b = rng.randint(10, 99), rng.randint(10, 99)
    answerability_data.append({
        "text": f"What is {a} + {b}?",
        "answerable": True,
        "correct_answer": str(a + b),
    })

# Unanswerable: fabricated countries
for name, region in [
    ("Veltharion", "South Pacific"), ("Krandosia", "Central Asia"),
    ("Melvithra", "Northern Europe"), ("Toravek", "West Africa"),
    ("Phaelundra", "South America"),
]:
    answerability_data.append({
        "text": f"What is the current population of {name}, the nation in {region}?",
        "answerable": False,
        "correct_answer": "UNANSWERABLE",
    })

# Unanswerable: contradictions
for c in [
    "What color is the invisible red ball?",
    "How many sides does a circular triangle have?",
    "What is the weight of a massless 5kg object?",
    "At what time does the event that never happens begin?",
    "What is the name of the unnamed theorem?",
]:
    answerability_data.append({
        "text": c,
        "answerable": False,
        "correct_answer": "UNANSWERABLE",
    })

# -- Error detection items --------------------------------------------------
error_det_items = []
for _ in range(15):
    a, b = rng.randint(100, 999), rng.randint(10, 99)
    correct = a * b
    has_error = rng.random() < 0.5
    if has_error:
        offset = rng.randint(1, max(1, correct // 50))
        presented = correct + rng.choice([-1, 1]) * offset
    else:
        presented = correct
    error_det_items.append({
        "question": f"What is {a} times {b}?",
        "presented_answer": str(presented),
        "has_error": has_error,
        "correct_answer": str(correct),
    })

print(f"Calibration items:    {len(calibration_data)}")
print(f"Answerability items:  {len(answerability_data)} "
      f"({sum(1 for q in answerability_data if q['answerable'])} answerable, "
      f"{sum(1 for q in answerability_data if not q['answerable'])} unanswerable)")
print(f"Error-detection items: {len(error_det_items)} "
      f"({sum(1 for i in error_det_items if i['has_error'])} with planted errors)")

## Helpers: Robust parsing & metrics

These work even if Pydantic structured output isn't supported by the model.
We try structured output first, then fall back to regex extraction from free-form text.

In [ ]:
def normalize_number(text):
    """Extract a signed integer from text. Returns string or ''."""
    if text is None:
        return ""
    s = str(text).strip().replace(",", "").replace(" ", "")
    m = re.search(r"-?\d+", s)
    return m.group(0) if m else ""


def parse_confidence(text):
    """Extract a 0-100 confidence number from free text, default 50."""
    if text is None:
        return 50
    m = re.search(r"(?:confidence|certainty)[^\d]*(\d{1,3})", str(text), re.I)
    if m:
        v = int(m.group(1))
        return max(0, min(100, v))
    # fallback: last percentage in the string
    m = re.search(r"(\d{1,3})\s*%", str(text))
    if m:
        return max(0, min(100, int(m.group(1))))
    return 50


def parse_yes_no(text, key_true="yes", key_false="no"):
    """Return True if text says yes/answerable/error, False otherwise."""
    if text is None:
        return False
    t = str(text).lower()
    # Prefer explicit markers
    if re.search(r"\b(yes|true|correct|answerable|has error|error exists)\b", t):
        return True
    if re.search(r"\b(no|false|unanswerable|cannot|unable|no error|correct)\b", t):
        return False
    return False


def safe_prompt(llm, prompt_text):
    """Call llm.prompt with error handling. Returns string response or ''."""
    try:
        return str(llm.prompt(prompt_text))
    except Exception as e:
        print(f"  [prompt error: {e}]")
        return ""


def compute_ece(confidences, accuracies, n_bins=10):
    """Expected Calibration Error over n_bins equal-width confidence bins."""
    total = len(confidences)
    if total == 0:
        return 0.0
    ece = 0.0
    width = 100.0 / n_bins
    for i in range(n_bins):
        lo = i * width
        hi = (i + 1) * width
        bucket = [(c, a) for c, a in zip(confidences, accuracies)
                  if (lo <= c < hi) or (i == n_bins - 1 and c == hi)]
        if bucket:
            avg_conf = sum(c for c, _ in bucket) / len(bucket) / 100.0
            avg_acc = sum(a for _, a in bucket) / len(bucket)
            ece += (len(bucket) / total) * abs(avg_acc - avg_conf)
    return ece

## Pydantic schemas (used when structured output is supported)

In [ ]:
if PYDANTIC_OK:
    class CalibratedAnswer(BaseModel):
        answer: str = Field(description="Just the final numeric answer.")
        confidence: int = Field(description="Confidence 0 to 100.", ge=0, le=100)

    class AnswerabilityJudgment(BaseModel):
        is_answerable: bool = Field(description="True if the question can be answered.")
        answer: Optional[str] = Field(default=None, description="If answerable, the answer.")

    class ErrorDetectionResult(BaseModel):
        has_error: bool = Field(description="True if the presented answer is wrong.")

    print("Pydantic schemas defined.")
else:
    print("Pydantic not available — will use text parsing fallback.")

## Task 1: Confidence Calibration

Ask model to answer + rate confidence, measure Expected Calibration Error (ECE).

In [ ]:
def ask_calibration(llm, question_text, correct_answer):
    """Get model answer + confidence. Returns (is_correct, confidence)."""
    prompt = (
        f"Solve this problem and rate your confidence from 0 to 100 that your "
        f"answer is correct.\n\nProblem: {question_text}\n\n"
        f"Respond as JSON: {{\"answer\": <number>, \"confidence\": <0-100>}}"
    )
    # Try structured output first
    if PYDANTIC_OK:
        try:
            r = llm.prompt(prompt, schema=CalibratedAnswer)
            ans = normalize_number(r.answer)
            conf = int(r.confidence)
            return ans == normalize_number(correct_answer), conf
        except Exception:
            pass
    # Fallback: free-form + parse
    text = safe_prompt(llm, prompt)
    ans = normalize_number(text)
    conf = parse_confidence(text)
    return ans == normalize_number(correct_answer), conf


def run_calibration(llm, questions):
    confidences, accuracies = [], []
    by_diff = {}
    for i, q in enumerate(questions):
        try:
            if KBENCH_OK:
                with kbench.chats.new("cal"):
                    correct, conf = ask_calibration(llm, q["text"], q["correct_answer"])
            else:
                correct, conf = ask_calibration(llm, q["text"], q["correct_answer"])
        except Exception as e:
            print(f"  Q{i} error: {e}")
            correct, conf = False, 50
        confidences.append(conf)
        accuracies.append(1.0 if correct else 0.0)
        d = q["difficulty"]
        by_diff.setdefault(d, [0, 0, 0])  # correct, total, conf_sum
        by_diff[d][1] += 1
        by_diff[d][2] += conf
        if correct:
            by_diff[d][0] += 1

    ece = compute_ece(confidences, accuracies)
    acc = sum(accuracies) / max(1, len(accuracies))
    avg_conf = (sum(confidences) / max(1, len(confidences))) / 100.0

    print(f"  Overall accuracy: {acc:.2f}")
    print(f"  Avg confidence:   {avg_conf:.2f}")
    print(f"  ECE (lower=better): {ece:.4f}")
    for d in ["easy", "medium", "hard", "very_hard"]:
        if d in by_diff:
            c, t, s = by_diff[d]
            print(f"    {d:10s} acc={c/t:.2f}  conf={s/t/100:.2f}")
    return round(max(0.0, 1.0 - ece), 4)

## Task 2: Answerability Detection

In [ ]:
def ask_answerability(llm, question_text):
    prompt = (
        "Determine whether this question can be definitively answered. "
        "A question is UNANSWERABLE if it asks about something fabricated, "
        "is self-contradictory, lacks information, or asks about future events.\n\n"
        f"Question: {question_text}\n\n"
        "Respond as JSON: {\"is_answerable\": true/false}"
    )
    if PYDANTIC_OK:
        try:
            r = llm.prompt(prompt, schema=AnswerabilityJudgment)
            return bool(r.is_answerable)
        except Exception:
            pass
    text = safe_prompt(llm, prompt)
    # Parse: look for 'unanswerable' or 'cannot' first
    t = text.lower()
    if "unanswerable" in t or "cannot" in t or '"is_answerable": false' in t or "is_answerable: false" in t:
        return False
    if "answerable" in t or '"is_answerable": true' in t or "is_answerable: true" in t:
        return True
    # default: assume model tried to answer (answerable)
    return True


def run_answerability(llm, questions):
    tp = tn = fp = fn = 0
    for i, q in enumerate(questions):
        try:
            if KBENCH_OK:
                with kbench.chats.new("ans"):
                    said_answerable = ask_answerability(llm, q["text"])
            else:
                said_answerable = ask_answerability(llm, q["text"])
        except Exception as e:
            print(f"  Q{i} error: {e}")
            said_answerable = True
        if q["answerable"]:
            if said_answerable:
                tp += 1
            else:
                fn += 1
        else:
            if said_answerable:
                fp += 1
            else:
                tn += 1
    total = tp + tn + fp + fn
    acc = (tp + tn) / max(1, total)
    sens = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    print(f"  Accuracy:    {acc:.2f}")
    print(f"  Sensitivity: {sens:.2f}  (TP={tp} FN={fn})")
    print(f"  Specificity: {spec:.2f}  (TN={tn} FP={fp})")
    return round(acc, 4)

## Task 3: Error Self-Detection

In [ ]:
def ask_error_detection(llm, question, presented_answer):
    prompt = (
        "Check if the student's answer to this math problem is correct.\n\n"
        f"Problem: {question}\nStudent's answer: {presented_answer}\n\n"
        "Respond as JSON: {\"has_error\": true/false}"
    )
    if PYDANTIC_OK:
        try:
            r = llm.prompt(prompt, schema=ErrorDetectionResult)
            return bool(r.has_error)
        except Exception:
            pass
    text = safe_prompt(llm, prompt).lower()
    if '"has_error": true' in text or "has_error: true" in text or "incorrect" in text or "wrong" in text or "error" in text and "no error" not in text:
        return True
    return False


def run_error_detection(llm, items):
    tp = tn = fp = fn = 0
    for i, item in enumerate(items):
        try:
            if KBENCH_OK:
                with kbench.chats.new("err"):
                    said_error = ask_error_detection(llm, item["question"], item["presented_answer"])
            else:
                said_error = ask_error_detection(llm, item["question"], item["presented_answer"])
        except Exception as e:
            print(f"  Item {i} error: {e}")
            said_error = False
        if item["has_error"]:
            if said_error:
                tp += 1
            else:
                fn += 1
        else:
            if said_error:
                fp += 1
            else:
                tn += 1
    total = tp + tn + fp + fn
    acc = (tp + tn) / max(1, total)
    print(f"  Accuracy: {acc:.2f}  (TP={tp} TN={tn} FP={fp} FN={fn})")
    return round(acc, 4)

## Register Tasks & Benchmark with the SDK

These decorated functions are what Kaggle Benchmarks uses to create the benchmark
and its tasks. The decorators only register when `kaggle_benchmarks` imported successfully.

In [ ]:
if KBENCH_OK:
    @kbench.task(
        name="Confidence Calibration",
        description=(
            "Measures Expected Calibration Error (ECE) over procedurally "
            "generated math questions. Score = 1 - ECE (higher is better)."
        ),
    )
    def confidence_calibration_task(llm) -> float:
        score = run_calibration(llm, calibration_data)
        kbench.assertions.assert_true(
            0.0 <= score <= 1.0,
            expectation="Calibration score must be between 0 and 1."
        )
        return score

    @kbench.task(
        name="Answerability Detection",
        description=(
            "Accuracy at distinguishing answerable questions from "
            "unanswerable ones (fabricated entities, contradictions)."
        ),
    )
    def answerability_task(llm) -> float:
        score = run_answerability(llm, answerability_data)
        kbench.assertions.assert_true(
            0.0 <= score <= 1.0,
            expectation="Answerability score must be between 0 and 1."
        )
        return score

    @kbench.task(
        name="Error Self-Detection",
        description=(
            "Accuracy at detecting arithmetic errors in presented solutions."
        ),
    )
    def error_detection_task(llm) -> float:
        score = run_error_detection(llm, error_det_items)
        kbench.assertions.assert_true(
            0.0 <= score <= 1.0,
            expectation="Error detection score must be between 0 and 1."
        )
        return score

    @kbench.benchmark(
        name="MetaCog-Bench",
        description=(
            "Metacognition benchmark across 3 dimensions: confidence "
            "calibration (ECE), answerability detection, and error "
            "self-detection. All questions procedurally generated."
        ),
    )
    def metacog_bench(llm) -> float:
        print("=" * 60)
        print("Task 1: Confidence Calibration")
        print("=" * 60)
        cal = confidence_calibration_task.run(llm=llm).result

        print("\n" + "=" * 60)
        print("Task 2: Answerability Detection")
        print("=" * 60)
        ans = answerability_task.run(llm=llm).result

        print("\n" + "=" * 60)
        print("Task 3: Error Self-Detection")
        print("=" * 60)
        err = error_detection_task.run(llm=llm).result

        composite = 0.40 * cal + 0.35 * ans + 0.25 * err
        print("\n" + "=" * 60)
        print("METACOG-BENCH SUMMARY")
        print("=" * 60)
        print(f"  Calibration    (40%): {cal:.4f}")
        print(f"  Answerability  (35%): {ans:.4f}")
        print(f"  Error Detect.  (25%): {err:.4f}")
        print(f"  Composite Score:      {composite:.4f}")

        kbench.assertions.assert_true(
            0.0 <= composite <= 1.0,
            expectation="Composite score must be between 0 and 1."
        )
        return round(composite, 4)

    print("✓ Tasks and benchmark registered with the SDK.")
else:
    print("⚠ Skipping SDK registration — install the SDK and restart the kernel.")

## Run the Benchmark

This actually executes the benchmark and publishes it to Kaggle Benchmarks.

In [ ]:
if KBENCH_OK:
    try:
        result = metacog_bench.run(llm=kbench.llm)
        print(f"\n🏁 MetaCognition Score: {result.result}")
        print(f"   Passed: {result.passed}")
    except Exception as e:
        print(f"Benchmark run error: {e}")
        print("The benchmark is still registered — you may be able to find it on your profile.")
else:
    print("Cannot run — SDK not available.")

## (Optional) Compare Across Models

In [ ]:
if KBENCH_OK:
    try:
        available = list(kbench.llms.keys())
        print("Available models:")
        for n in available:
            print(f"  - {n}")
    except Exception as e:
        print(f"Could not list models: {e}")

In [ ]:
# Optional: run across a few models to show discriminatory power.
# Comment out to skip — the benchmark is already created by the cell above.
RUN_MULTI_MODEL = False  # Set True to enable

if KBENCH_OK and RUN_MULTI_MODEL:
    results = {}
    try:
        names = list(kbench.llms.keys())[:3]
        for name in names:
            print(f"\n### Running on {name} ###")
            try:
                r = metacog_bench.run(llm=kbench.llms[name])
                results[name] = r.result
            except Exception as e:
                print(f"  {name} failed: {e}")
                results[name] = None
        print("\n=== Results ===")
        for n, s in sorted(results.items(), key=lambda x: (x[1] or -1), reverse=True):
            print(f"  {n:30s} {s}")
    except Exception as e:
        print(f"Multi-model run error: {e}")